## Merge and Explore 

#### **Merge the data that contains the speeches with the data that contains the speaker data such as party** 

Then, explore: 

General overview of the data - structure, counts, initial analysis. What do the speeches look like? 

Cleaning the data - getting it in better shape, ready for analysis. 

In [1]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import gc

RAW = Path('../data/raw')
DERIVED = Path('../data/derived')
DERIVED.mkdir(exist_ok=True)
pd.__version__

'3.1.1'

## Strategy

To avoid duplicating the 7-8 gb of data by combining it all into one file, we can just merge on the fly in duckdb with each query. 

In [2]:
#establish database s
con = duckdb.connect(str(DERIVED / 'congress.duckdb'))

con.execute(f'''
CREATE OR REPLACE VIEW speeches AS
SELECT * FROM read_parquet('{RAW}/us_congressional_speeches/data/*.parquet')
''')

con.execute(f'''
CREATE OR REPLACE VIEW speakermap AS
SELECT * FROM read_parquet('{DERIVED}/speakermap.parquet')
''')

#creates two tables without duplicating the data, and makes the party group variable as well. 

display(con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT speech_id) AS n_speech_ids,
       MIN(speech_id // 10000000) AS first_congress,
       MAX(speech_id // 10000000) AS last_congress
FROM speeches
''').df())
display(
con.sql('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT speech_id) AS n_speech_ids,
       MIN(speech_id // 10000000) AS first_congress,
       MAX(speech_id // 10000000) AS last_congress
FROM speakermap
''').df()
)

#no need to look at the text yet because it is huge

,n_rows,n_speech_ids,first_congress,last_congress
0,17395884,17395884,43,111


,n_rows,n_speech_ids,first_congress,last_congress
0,13311255,13311255,43,111


Compare to numbers from 03_speakers.ipynb. every row unique, but there are several million more rows than there were in the speaker map. 

#### Merge

In [3]:
#merge on speech_id

#first, list file columns 
first_map = pd.read_csv(RAW / 'speakermap' / '043_SpeakerMap.txt', sep='|', quoting=3, dtype=str)
print(f'Map Column Count: {len(first_map.columns)}')
print(first_map.columns)
print(first_map['speech_id'].min())
print(first_map['speech_id'].max())
first_text = pd.read_parquet(RAW / 'us_congressional_speeches' / 'data' / 'train-00000-of-00028.parquet')
print(f'Text File Column Count: {len(first_text.columns)}')
print(first_text.columns)
del(first_text)
gc.collect() #remove big file from memory


Map Column Count: 10
Index(['speakerid', 'speech_id', 'lastname', 'firstname', 'chamber', 'state',
       'gender', 'party', 'district', 'nonvoting'],
      dtype='str')
430000002
430119301
Text File Column Count: 15
Index(['speech_id', 'text', 'chamber', 'date', 'number_within_file', 'speaker',
       'first_name', 'last_name', 'state', 'gender', 'line_start', 'line_end',
       'file', 'char_count', 'word_count'],
      dtype='str')


0

We have 10 columns in each speakermap, and 15 in each text file. There are several columns that overlap. They are speech id (the key), state, gender, and chamber
I don't know how overlap there is between them, or if they measure the same thing. I assume that at least chamber is worth keeping. Others can probably be tossed. 

*There is also the party_group variable that I made and is in the joined parquet and the database file*

Also, per this test and the codebook, the speech_id begins with the congress # as the first two or three digits, and the last seven are the id. We can divide to get the congress #. 

We will use a left merge on speeches to keep every speech. 

In [4]:
con.execute('''
CREATE OR REPLACE VIEW speeches_merged AS
SELECT speech.*, -- all columns 
       speech_id // 10000000 AS congress,
       map.* -- all columns
FROM speeches AS speech
LEFT JOIN speakermap AS map USING (speech_id)
''')

#easier to just name columns rather than all and get rid of duplicates later. 

con.execute('''
CREATE OR REPLACE VIEW speeches_merged AS
SELECT speech.*,
       speech_id // 10000000 AS congress,
       map.speakerid,
       map.lastname,
       map.firstname,
       map.party AS party,
       map.district,
       map.nonvoting,
       map.party_group
FROM speeches AS speech
LEFT JOIN speakermap AS map USING (speech_id)
''')

display(
    con.sql('''
       SELECT COUNT(*) AS rows, COUNT(party) AS rows_party
       FROM speeches_merged
       ''').df()
)

con.sql('DESCRIBE speeches_merged').df()[['column_name', 'column_type']]

,rows,rows_party
0,17395884,13309706


,column_name,column_type
0,speech_id,BIGINT
1,text,VARCHAR
2,chamber,VARCHAR
3,date,TIMESTAMP
4,number_within_file,BIGINT
5,speaker,VARCHAR
6,first_name,VARCHAR
7,last_name,VARCHAR
8,state,VARCHAR
9,gender,VARCHAR


#### Percentage of each congress's speeches that are missing the party variable
This could be an indicator that many of these are not really speeches but could be procedural junk. 

In [5]:
con.sql('''
SELECT congress,
       COUNT(*) as rows,
       COUNT(*) - COUNT(party) as missing_party_count,
       ROUND(100.0 * (COUNT(*) - COUNT(party)) / COUNT(*), 2) AS missing_share
FROM speeches_merged
GROUP BY congress
ORDER BY missing_share DESC
LIMIT 10
''')

congressBIGINT,rowsBIGINT,missing_party_countBIGINT,missing_shareDOUBLE
110,212386,79204,37.29
111,179268,66718,37.22
109,186708,67202,35.99
108,194544,68880,35.41
107,178667,62252,34.84
105,209479,69422,33.14
101,250363,80846,32.29
106,208086,66679,32.04
102,239303,74672,31.2
103,235325,72586,30.85


In [6]:
con.sql('''
SELECT congress,
       COUNT(*) as rows,
       COUNT(*) - COUNT(party) as missing_party_count,
       ROUND(100.0 * (COUNT(*) - COUNT(party)) / COUNT(*), 2) AS missing_share
FROM speeches_merged
GROUP BY congress
ORDER BY missing_share
LIMIT 10
''')

congressBIGINT,rowsBIGINT,missing_party_countBIGINT,missing_shareDOUBLE
55,195024,30572,15.68
61,279720,45199,16.16
72,237071,40097,16.91
82,261703,44777,17.11
74,247172,43567,17.63
57,134113,23667,17.65
71,321499,57132,17.77
60,155293,27707,17.84
73,195779,35112,17.93
75,253399,46017,18.16


In [7]:
con.execute('''
CREATE OR REPLACE TABLE everything_but_speeches AS
SELECT * EXCLUDE (text)
FROM speeches_merged
''')

#this is a file in memory, but it is missing the text so it is not nearly as large. For all analysis that doesn't involve the speeches themselves, like counts. This file is about 500 mb. 

In [8]:
con.close()